# ScalogramV2 Grad-CAM Visualization
This notebook demonstrates how to visualize the model's focus on the PC3/ULF band before a seismic event using Grad-CAM.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import sys
import os
from pathlib import Path

# Add root to path
ROOT_DIR = Path(os.getcwd()).parent
sys.path.append(str(ROOT_DIR))

from model.mtl_crnn import ScalogramV2Model
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

In [ ]:
# 1. Load Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ScalogramV2Model(pretrained=False).to(device)
# Placeholder: User would load their downloaded weights here
# checkpoint = torch.load('../models/best_model.pth', map_location=device)
# model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

In [ ]:
# 2. Prepare Grad-CAM Wrapper
class GradCAMModelWrapper(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
    def forward(self, x):
        out_b, _, _ = self.base_model(x)
        return out_b

wrapper = GradCAMModelWrapper(model)
target_layers = [model.backbone.features[-1]]
cam = GradCAM(model=wrapper, target_layers=target_layers)

In [ ]:
# 3. Run on Sample Tensor
# Assuming a sample tensor shape [3, 128, 1440]
dummy_tensor = torch.randn(1, 3, 128, 1440).to(device)
targets = [ClassifierOutputTarget(1)] # Target 'Precursor' class

grayscale_cam = cam(input_tensor=dummy_tensor, targets=targets)[0, :]

# Visualization
plt.figure(figsize=(15, 5))
plt.imshow(grayscale_cam, aspect='auto', cmap='jet')
plt.title("Model Attention Heatmap (Grad-CAM)")
plt.colorbar()
plt.show()